# Underwater 3D Reconstruction Pipeline

**Complete end-to-end pipeline for underwater 3D reconstruction using Gaussian Splatting**

This notebook consolidates the main3 branch functionality into a unified, executable workflow covering:

1. Dataset exploration and quality assessment
2. Image classification and ranking with CLIP
3. Preprocessing and artifact removal
4. Template matching for object detection
5. WaterSplatting 3D reconstruction preparation

---

## Table of Contents

1. [Environment Setup](#1-environment-setup)
2. [Dataset Download and Ranking](#2-dataset-download-and-ranking)
3. [Exploratory Data Analysis](#3-exploratory-data-analysis)
4. [Image Preprocessing](#4-image-preprocessing)
5. [Template Matching (Optional)](#5-template-matching-optional)
6. [3D Reconstruction Preparation](#6-3d-reconstruction-preparation)
7. [Summary and Next Steps](#7-summary-and-next-steps)

---

**Author**: Lockheed Martin NAISS Team  
**Date**: March 2026  
**Branch**: main3 (consolidated from image_class, watersplatting_initial, optical-eda)

## 1. Environment Setup

Install required dependencies and verify the environment is correctly configured.

In [ ]:
# Check Python version
import sys
print(f"Python version: {sys.version}")
print(f"Python executable: {sys.executable}")

# Core imports
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("\nCore libraries loaded successfully!")

In [ ]:
# Check for PyTorch and CUDA availability
try:
    import torch
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nUsing device: {device}")
except ImportError:
    print("PyTorch not installed. Install with: pip install torch torchvision")
    device = "cpu"

In [ ]:
# Check for CLIP (for image ranking)
try:
    from transformers import CLIPModel, CLIPProcessor
    print("CLIP (transformers) loaded successfully!")
    CLIP_AVAILABLE = True
except ImportError:
    print("CLIP not available. Install with: pip install transformers")
    CLIP_AVAILABLE = False

# Check for OpenCV
try:
    import cv2
    print(f"OpenCV version: {cv2.__version__}")
    OPENCV_AVAILABLE = True
except ImportError:
    print("OpenCV not available. Install with: pip install opencv-python")
    OPENCV_AVAILABLE = False

### Configure Project Paths

Set up the working directories for the pipeline.

In [ ]:
# Project root directory
PROJECT_ROOT = Path("/Users/priyanshudey/Code/Lockheed_Sonar")
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
DATA_DIR = PROJECT_ROOT / "data"  # Create this directory for datasets
OUTPUT_DIR = PROJECT_ROOT / "pipeline_outputs"  # Pipeline results

# Create directories if they don't exist
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Scripts directory: {SCRIPTS_DIR}")
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

# Add scripts to Python path for imports
sys.path.insert(0, str(SCRIPTS_DIR))

---

## 2. Dataset Download and Ranking

Download underwater imagery datasets from Harvard Dataverse and rank images using CLIP to identify the best candidates for 3D reconstruction.

### What is CLIP Ranking?

CLIP (Contrastive Language-Image Pretraining) is an AI model that understands both images and text. We use it to:
- Score images based on how well they match desired characteristics (shipwrecks, objects, structure)
- Penalize empty/boring images (open water, blur, darkness)
- Keep only the top K most promising images for reconstruction

This saves disk space and focuses computational resources on high-quality frames.

In [ ]:
# Configuration for dataset download and ranking
DATASET_DOI = "doi:10.7910/DVN/VZD5S6"  # Harvard Dataverse underwater dataset
TOP_K_IMAGES = 200  # Number of top images to keep
BATCH_SIZE = 16  # Images to process at once (increase for GPU)
RANKED_OUTPUT = OUTPUT_DIR / "ranked_dataset"

print(f"Dataset DOI: {DATASET_DOI}")
print(f"Will keep top {TOP_K_IMAGES} images")
print(f"Batch size: {BATCH_SIZE}")
print(f"Output directory: {RANKED_OUTPUT}")

In [ ]:
# Option 1: Run download_and_rank.py as a subprocess
# This is safer and uses the fully tested script

if CLIP_AVAILABLE:
    print("Starting incremental download and ranking...")
    print("This will:")
    print("  1. Download archives from Harvard Dataverse")
    print("  2. Extract and score images with CLIP")
    print("  3. Keep only top-K images, delete rest")
    print("  4. Move to next archive")
    print("\nThis may take 30-60 minutes depending on dataset size and internet speed.")
    print("\n" + "="*60)
    
    # Uncomment to run:
    # !python {SCRIPTS_DIR}/dataset_tools/download_and_rank.py \
    #     --outdir {RANKED_OUTPUT} \
    #     --k {TOP_K_IMAGES} \
    #     --device {device} \
    #     --batch-size {BATCH_SIZE}
    
    print("\n[SKIPPED] Uncomment the above command to run download and ranking.")
    print("For demo purposes, we'll proceed with mock data or existing datasets.")
else:
    print("CLIP not available. Skipping download and ranking.")
    print("Install transformers library to enable this feature.")

### Check Ranking Results

If you ran the download and ranking, inspect the results:

In [ ]:
# Check if ranking results exist
ranking_summary = RANKED_OUTPUT / "results" / "summary.json"

if ranking_summary.exists():
    with open(ranking_summary, 'r') as f:
        summary = json.load(f)
    
    print("="*60)
    print("RANKING SUMMARY")
    print("="*60)
    
    metadata = summary['metadata']
    print(f"\nTotal images processed: {metadata['processed_images']:,}")
    print(f"Failed images: {metadata['failed_images']:,}")
    print(f"Deleted (not in top-{metadata['k']}): {metadata['deleted_images']:,}")
    print(f"\nFinal top-K count: {metadata['final_top_k_count']}")
    
    # Show top 5 images
    print(f"\nTop 5 Images:")
    for img in summary['ranked_images'][:5]:
        print(f"  Rank {img['rank']}: {img['relative_path']}")
        print(f"    Score: {img['score']:.4f}")
        print(f"    CLIP+: {img['score_components']['clip_positive']:.4f}, CLIP-: {img['score_components']['clip_negative']:.4f}")
else:
    print("No ranking results found. Run download_and_rank.py first or use existing dataset.")

---

## 3. Exploratory Data Analysis

Analyze the downloaded/ranked dataset to understand:
- Image quality (blur, contrast, sharpness)
- Color profiles (underwater blue/green cast)
- Scene content (coral, rocks, objects, open water)
- COLMAP reconstruction quality
- Gaussian Splatting readiness

This comprehensive analysis generates an interactive HTML report.

In [ ]:
# Configuration for EDA
# Point this to your dataset directory
# Option 1: Use ranked dataset from step 2
# Option 2: Use an existing dataset

# For demo, use a sample path (update this to your actual dataset)
DATASET_PATH = RANKED_OUTPUT / "images" if (RANKED_OUTPUT / "images").exists() else DATA_DIR / "sample_dataset"

print(f"Dataset path for EDA: {DATASET_PATH}")
print(f"Exists: {DATASET_PATH.exists()}")

if not DATASET_PATH.exists():
    print("\nWARNING: Dataset path does not exist.")
    print("Please update DATASET_PATH to point to a valid underwater image dataset.")

In [ ]:
# Run comprehensive EDA using optical_imagery_eda.py
if OPENCV_AVAILABLE and DATASET_PATH.exists():
    print("Running comprehensive underwater dataset EDA...")
    print("This will analyze:")
    print("  - Image metadata (dimensions, file sizes)")
    print("  - Color profiles (blue/red ratio, brightness)")
    print("  - Quality metrics (blur, contrast, edge density)")
    print("  - Scene content (coral, rocks, open water)")
    print("  - COLMAP reconstruction data (if available)")
    print("\nProcessing time: ~100-200ms per image")
    print("\n" + "="*60)
    
    # Uncomment to run:
    # !python {SCRIPTS_DIR}/eda/optical_imagery_eda.py {DATASET_PATH}
    
    print("\n[SKIPPED] Uncomment the above command to run full EDA.")
    print("Output will be saved as: underwater_eda_report.html")
else:
    print("Skipping EDA:")
    if not OPENCV_AVAILABLE:
        print("  - OpenCV not available")
    if not DATASET_PATH.exists():
        print("  - Dataset path does not exist")

### Quick Dataset Statistics

If you have a dataset, let's compute some quick statistics without the full EDA:

In [ ]:
if DATASET_PATH.exists():
    # Count images
    image_extensions = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}
    image_files = []
    
    for ext in image_extensions:
        image_files.extend(list(DATASET_PATH.rglob(f"*{ext}")))
        image_files.extend(list(DATASET_PATH.rglob(f"*{ext.upper()}")))
    
    print(f"Found {len(image_files)} images in {DATASET_PATH}")
    
    if len(image_files) > 0:
        # Sample 10 images for quick stats
        sample_size = min(10, len(image_files))
        sample_images = np.random.choice(image_files, sample_size, replace=False)
        
        widths, heights, sizes = [], [], []
        
        for img_path in sample_images:
            try:
                with Image.open(img_path) as img:
                    w, h = img.size
                    widths.append(w)
                    heights.append(h)
                    sizes.append(img_path.stat().st_size)
            except Exception as e:
                print(f"Error reading {img_path.name}: {e}")
        
        if widths:
            print(f"\nSample Statistics (n={len(widths)}):")
            print(f"  Resolution: {min(widths)}x{min(heights)} to {max(widths)}x{max(heights)}")
            print(f"  File size: {min(sizes)/1024:.1f} KB to {max(sizes)/1024:.1f} KB")
            print(f"  Avg file size: {np.mean(sizes)/1024:.1f} KB")
else:
    print("No dataset available for analysis.")

---

## 4. Image Preprocessing

Clean the dataset by:
1. **Analyzing**: CLIP classification to identify empty/useless images
2. **Detecting**: Static border artifacts (camera housing, overlays)
3. **Cleaning**: Remove empty images, crop artifacts, convert TIFF to PNG

### Why Preprocessing?

3D reconstruction algorithms struggle with:
- Empty frames (open water, nothing to match)
- Static artifacts (camera housing appears in every frame → false matches)
- Blurry/dark images (no discernible features)

Preprocessing improves reconstruction quality by removing these problematic frames.

In [ ]:
# Preprocessing configuration
ANALYSIS_OUTPUT = OUTPUT_DIR / "classification_output"
CLEANED_OUTPUT = OUTPUT_DIR / "cleaned_datasets"

print(f"Analysis output: {ANALYSIS_OUTPUT}")
print(f"Cleaned output: {CLEANED_OUTPUT}")

### Phase 1: Analyze Dataset

In [ ]:
# Run preprocessing analysis
# Note: This requires CLIP to be installed

if CLIP_AVAILABLE and DATASET_PATH.exists():
    print("Running preprocessing analysis...")
    print("This will:")
    print("  1. Classify each image using CLIP (12 underwater categories)")
    print("  2. Detect static border artifacts (camera housing)")
    print("  3. Generate classification CSV and summary JSON")
    print("\n" + "="*60)
    
    # Uncomment to run:
    # !python {SCRIPTS_DIR}/preprocessing/preprocess_datasets.py analyze \
    #     --outdir {ANALYSIS_OUTPUT} \
    #     --device {device} \
    #     --batch-size {BATCH_SIZE}
    
    print("\n[SKIPPED] Uncomment the above command to run analysis.")
    print("Note: Update dataset paths in preprocess_datasets.py DATASET_CONFIGS first.")
else:
    print("Skipping preprocessing analysis:")
    if not CLIP_AVAILABLE:
        print("  - CLIP not available")
    if not DATASET_PATH.exists():
        print("  - Dataset path does not exist")

### Phase 2: Clean Dataset

In [ ]:
# Run cleaning based on analysis results
analysis_summary = ANALYSIS_OUTPUT / "classification_summary.json"

if analysis_summary.exists():
    print("Running dataset cleaning...")
    print("This will:")
    print("  1. Remove images classified as 'empty' (open water, murky, blurry)")
    print("  2. Crop detected border artifacts")
    print("  3. Convert TIFF to PNG")
    print("  4. Preserve directory structure")
    print("\n" + "="*60)
    
    # Uncomment to run:
    # !python {SCRIPTS_DIR}/preprocessing/preprocess_datasets.py clean \
    #     --analysis-dir {ANALYSIS_OUTPUT} \
    #     --outdir {CLEANED_OUTPUT}
    
    print("\n[SKIPPED] Uncomment the above command to run cleaning.")
else:
    print("Analysis results not found. Run Phase 1 first.")

---

## 5. Template Matching (Optional)

If your dataset contains specific objects you want to isolate (e.g., shipwrecks, vehicles), use template matching to:
- Detect objects using a template image
- Crop regions of interest
- Focus 3D reconstruction on specific targets

This is useful when you have a specific inspection target rather than general terrain mapping.

In [ ]:
# Template matching example (sunboat shipwreck)
# You'll need:
#  1. A template image of the object
#  2. A dataset containing that object

print("Template matching scripts available:")
print(f"  - {SCRIPTS_DIR}/template_matching/sunboat_templatematching.py")
print(f"  - {SCRIPTS_DIR}/template_matching/vehicle_artifact_crop.py")
print(f"  - {SCRIPTS_DIR}/template_matching/sunboat_crop.py")
print("\nThese require specific datasets. See script documentation for usage.")

---

## 6. 3D Reconstruction Preparation

Prepare the cleaned dataset for WaterSplatting 3D reconstruction:
1. Convert images to COLMAP-ready scene
2. Run COLMAP to estimate camera poses
3. Train WaterSplatting model
4. View results

### Prerequisites

- WaterSplatting installed: `pip install -e .` (from repository root)
- Nerfstudio installed: `pip install nerfstudio==1.1.4`
- COLMAP installed (for pose estimation)

In [ ]:
# Check if WaterSplatting is installed
try:
    import water_splatting
    print(f"WaterSplatting version: {water_splatting.version.__version__ if hasattr(water_splatting, 'version') else 'unknown'}")
    WATERSPLATTING_AVAILABLE = True
except ImportError:
    print("WaterSplatting not installed.")
    print("Install with: pip install -e . (from repository root)")
    WATERSPLATTING_AVAILABLE = False

# Check for Nerfstudio CLI
import shutil
ns_train = shutil.which("ns-train")
if ns_train:
    print(f"Nerfstudio CLI found: {ns_train}")
    NERFSTUDIO_AVAILABLE = True
else:
    print("Nerfstudio CLI not found.")
    print("Install with: pip install nerfstudio")
    NERFSTUDIO_AVAILABLE = False

### Create COLMAP Scene

In [ ]:
# Use the create_valid_scene.py script to convert images to a COLMAP-ready scene
SCENE_OUTPUT = OUTPUT_DIR / "watersplatting_data" / "scene1"

print("To create a COLMAP scene:")
print(f"  1. Place cleaned images in a directory")
print(f"  2. Run: python 'Download Datasets/create_valid_scene.py'")
print(f"  3. Follow prompts to specify:")
print(f"     - Image directory path")
print(f"     - Output scene name")
print(f"     - Train/eval split ratio")
print(f"\nOutput will be saved to: ../watersplatting_data/{{dataset}}/{{scene}}/")
print(f"\nScene structure:")
print(f"  scene/")
print(f"    images/        # All images")
print(f"    sparse/0/      # COLMAP reconstruction (cameras.bin, images.bin, points3D.bin)")
print(f"    poses_bounds.npy  # Camera poses in LLFF format")

### Train WaterSplatting Model

In [ ]:
if NERFSTUDIO_AVAILABLE and WATERSPLATTING_AVAILABLE:
    print("To train a WaterSplatting model:")
    print("\n1. Basic training command:")
    print(f"   ns-train water-splatting --data {{scene_path}}")
    print("\n2. Example with specific scene:")
    print(f"   ns-train water-splatting --data ../watersplatting_data/underwater/scene1")
    print("\n3. With custom output directory:")
    print(f"   ns-train water-splatting --data {{scene_path}} --output-dir {OUTPUT_DIR}/models/run1")
    print("\nTraining typically takes 30-60 minutes on a modern GPU.")
    print("\nUncomment below to run training:")
    
    # Uncomment and update scene_path to run:
    # scene_path = "../watersplatting_data/underwater/scene1"
    # !ns-train water-splatting --data {scene_path} --output-dir {OUTPUT_DIR}/models/run1
else:
    print("WaterSplatting or Nerfstudio not available.")
    print("Install both to enable 3D reconstruction.")

### View Results

In [ ]:
if NERFSTUDIO_AVAILABLE:
    print("To view the trained model:")
    print("\n1. Find the config file:")
    print(f"   {OUTPUT_DIR}/models/run1/water-splatting/{{timestamp}}/config.yml")
    print("\n2. Launch viewer:")
    print("   ns-viewer --load-config {{config_path}}")
    print("\n3. Open browser to:")
    print("   http://localhost:7007")
    print("\nFor remote servers, use SSH port forwarding:")
    print("   ssh -L 7007:localhost:7007 user@server")
else:
    print("Nerfstudio viewer not available.")

---

## 7. Summary and Next Steps

This notebook provided a complete workflow for underwater 3D reconstruction:

### What We Covered

1. **Environment Setup**: Verified dependencies and configured paths
2. **Dataset Download & Ranking**: Used CLIP to select top images from Harvard Dataverse
3. **EDA**: Analyzed image quality, color, content, and 3DGS readiness
4. **Preprocessing**: Removed empty images and static artifacts
5. **Template Matching**: (Optional) Object detection and isolation
6. **3D Reconstruction**: Prepared scenes for WaterSplatting

### Next Steps

#### For Research
- **Experiment with CLIP weights**: Adjust scoring to prioritize different features
- **Compare datasets**: Run EDA on multiple datasets to identify best candidates
- **Tune preprocessing**: Adjust artifact detection thresholds

#### For Production
- **Automate pipeline**: Chain scripts together with bash/Python
- **Monitor quality**: Track EDA metrics over time
- **Optimize performance**: Use GPU batching, parallel processing

#### For Deployment
- **Docker container**: Package entire environment
- **CI/CD pipeline**: Automated testing and validation
- **Cloud deployment**: Scale to process large datasets

### Key Resources

- **Repository**: /Users/priyanshudey/Code/Lockheed_Sonar
- **Documentation**: README.md, MERGE_REPORT.md
- **Scripts**: scripts/ directory (well-commented, production-ready)
- **Papers**: 
  - WaterSplatting: https://arxiv.org/pdf/2408.08206
  - CLIP: https://arxiv.org/abs/2103.00020
  - 3D Gaussian Splatting: https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/

### Troubleshooting

**Common Issues**:

1. **CUDA out of memory**: Reduce batch size
2. **COLMAP fails**: Check image quality, ensure sufficient overlap
3. **Import errors**: Verify all dependencies installed
4. **Slow processing**: Use GPU, increase batch size

**Getting Help**:
- Check script documentation (all scripts have detailed docstrings)
- Review MERGE_REPORT.md for branch integration details
- See README.md for full installation guide

In [ ]:
# Generate pipeline execution summary
print("="*80)
print("PIPELINE EXECUTION SUMMARY")
print("="*80)

print("\nEnvironment:")
print(f"  Python: {sys.version.split()[0]}")
print(f"  Device: {device if 'device' in locals() else 'N/A'}")
print(f"  CLIP available: {CLIP_AVAILABLE if 'CLIP_AVAILABLE' in locals() else 'N/A'}")
print(f"  OpenCV available: {OPENCV_AVAILABLE if 'OPENCV_AVAILABLE' in locals() else 'N/A'}")
print(f"  WaterSplatting available: {WATERSPLATTING_AVAILABLE if 'WATERSPLATTING_AVAILABLE' in locals() else 'N/A'}")

print("\nDirectories:")
print(f"  Project root: {PROJECT_ROOT}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Dataset path: {DATASET_PATH if 'DATASET_PATH' in locals() else 'N/A'}")

print("\nPipeline Components:")
components = [
    ("1. Download & Ranking", "download_and_rank.py"),
    ("2. EDA", "optical_imagery_eda.py"),
    ("3. Preprocessing", "preprocess_datasets.py"),
    ("4. Template Matching", "template_matching/*.py"),
    ("5. Scene Creation", "create_valid_scene.py"),
    ("6. Training", "ns-train water-splatting"),
]

for name, script in components:
    print(f"  {name}: {script}")

print("\n" + "="*80)
print("✅ Notebook execution complete!")
print("="*80)
print("\nTo run the full pipeline:")
print("  1. Uncomment and run the download & ranking cell")
print("  2. Run EDA on the ranked dataset")
print("  3. Run preprocessing (analyze + clean)")
print("  4. Create COLMAP scene from cleaned images")
print("  5. Train WaterSplatting model")
print("  6. View results in browser")
print("\nEach step can be run independently or as part of the full workflow.")